In [ ]:
# Install required libraries
!pip install pandas numpy matplotlib --quiet

# Exercise - Third-Party Vendor Risk Scoring

# PrecisionRisk AI: Vendor Risk Assessment & SLA Compliance Dashboard

This notebook implements vendor risk scoring and SLA compliance monitoring for PrecisionRisk AI's 5 AI vendor relationships, aligned with **Malaysian regulatory requirements** (BNM RMiT, AIGE Guidelines, PDPA 2024).

---

## Quick Reference: Key Concepts

**Vendor Risk Scoring Criteria**:
- **Security Posture** (30%): Certifications (SOC2, ISO27001), encryption, access controls
- **Data Handling** (25%): Data residency, retention policies, processing agreements
- **Transparency** (20%): Model explainability, bias testing, documentation
- **Financial Stability** (15%): Revenue, market position, business continuity
- **SLA Quality** (10%): Uptime guarantees, response times, penalty structures

**Risk Score Scale**: 1 (Lowest Risk) → 5 (Highest Risk)
- 1.0-1.5: Highly Recommended - Strong across all dimensions
- 1.5-2.5: Recommended - Good overall with minor gaps
- 2.5-3.5: Acceptable with Monitoring - Notable gaps requiring attention
- 3.5-5.0: Not Recommended - Significant governance concerns

**Vendor Governance Best Practices**:
- Maintain 2+ active vendor relationships for redundancy (BNM RMiT requirement)
- Review vendor risk quarterly; SLA compliance monthly
- Test failover procedures at least annually
- Require 7+ days advance notice for model updates
- Ensure vendors comply with PDPA 2024 - processors now directly liable

## Scenario

PrecisionRisk AI is a Kuala Lumpur-based InsurTech licensed by BNM, using GenAI for claims processing, fraud detection, and customer chatbots across both conventional insurance and takaful operators. After a vendor model update caused false claim denials - triggering potential NAIC/BNM regulatory exposure - the CRO wants formal risk scoring and SLA monitoring across all 5 vendors.


## Step 1: Setup and Vendor Data

In [ ]:
# =============================================================
# VENDOR EVALUATION SCORES (0-100 per criterion)
# Based on completed Vendor Assessment (vendor_governance_starter.xlsx)
# Updated to PrecisionRisk AI vendor names - Malaysian context
# =============================================================
import pandas as pd

vendors_df = pd.DataFrame({
    'Vendor': ['ClarityCore API', 'ArcVault AI', 'NexaScale AI', 'PrismLogic AI', 'SentinelIQ AI'],
    'Security_Score':       [85, 95, 95, 70, 90],
    'Data_Handling_Score':  [80, 90, 85, 60, 85],
    'Model_Transparency':   [90, 70, 70, 45, 90],   # PrismLogic flagged LOW
    'Financial_Stability':  [85, 95, 100, 70, 88],
    'SLA_Compliance':       [85, 90, 95, 60, 85],
    'Exit_Strategy':        [90, 95, 90, 55, 92],   # PrismLogic PARTIAL exit
    'AI_Model_Governance':  [85, 80, 90, 60, 85],
})

print(f"Loaded {len(vendors_df)} vendors with {len(vendors_df.columns)-1} criteria each.")
vendors_df


In [ ]:
# =============================================================
# SLA PERFORMANCE DATA - 30-day tracking per vendor (simulated)
# Baselines derived from Vendor Assessment SLA commitments
# =============================================================
import numpy as np
import matplotlib.pyplot as plt
from math import pi

np.random.seed(42)

vendor_baselines = {
    'ClarityCore API': {'uptime_mean': 99.92, 'uptime_std': 0.03,  'latency_mean': 450, 'latency_std': 50},
    'ArcVault AI':     {'uptime_mean': 99.98, 'uptime_std': 0.01,  'latency_mean': 380, 'latency_std': 30},
    'NexaScale AI':    {'uptime_mean': 99.99, 'uptime_std': 0.005, 'latency_mean': 280, 'latency_std': 25},
    'PrismLogic AI':   {'uptime_mean': 99.51, 'uptime_std': 0.15,  'latency_mean': 920, 'latency_std': 200},  # poor performer
    'SentinelIQ AI':   {'uptime_mean': 99.93, 'uptime_std': 0.02,  'latency_mean': 430, 'latency_std': 40},
}

sla_records = []
for vendor, b in vendor_baselines.items():
    for day in range(1, 31):
        uptime  = min(100, max(98, np.random.normal(b['uptime_mean'],  b['uptime_std'])))
        latency = max(100, np.random.normal(b['latency_mean'], b['latency_std']))
        sla_records.append({'Vendor': vendor, 'Day': day,
                            'Uptime': round(uptime, 3), 'Latency_ms': round(latency, 1)})

sla_df = pd.DataFrame(sla_records)
print(f"SLA performance data: {len(sla_df)} records across {sla_df['Vendor'].nunique()} vendors")
sla_df.head(10)


## Step 2: Define Risk Scoring Weights

Vendor risk scoring requires weighting criteria by importance. This is a governance decision - the weights reflect what matters most for PrecisionRisk AI's Malaysian insurance and takaful context.

Weights are aligned with **BNM RMiT** outsourcing risk priorities and **AIGE** governance principles:
- Security is paramount (customer financial & takaful participant data)
- Data handling matters for PDPA 2024 compliance
- Transparency is critical for BNM claims explainability requirements
- Financial stability ensures vendor longevity
- SLA quality affects operational reliability
- AI Model Governance is critical for GenAI vendors (training data provenance, model update notification, fine-tuning portability)


In [ ]:
# Scoring weights - must sum to 1.0
# Aligned with BNM RMiT risk priorities for Malaysian InsurTech
scoring_weights = {
    'Security_Score':      0.25,   # Highest - financial & takaful participant data protection
    'Data_Handling_Score': 0.20,   # PDPA 2024 compliance; data residency for takaful
    'Model_Transparency':  0.15,   # BNM & AIGE explainability requirement for claims decisions
    'Financial_Stability': 0.10,   # Vendor longevity risk
    'SLA_Compliance':      0.10,   # Operational reliability
    'Exit_Strategy':       0.10,   # BNM RMiT: exit/migration must be tested
    'AI_Model_Governance': 0.10,   # Model update notification, bias testing, governance docs
}

total = sum(scoring_weights.values())
print(f"Total weight: {total:.2f} (must equal 1.00)")
assert abs(total - 1.0) < 0.01, f'Weights must sum to 1.0, got {total}'
print("\nWeight breakdown:")
for k, v in scoring_weights.items():
    print(f"  {k:<25} {v:.0%}")


## Step 3: Calculate Risk Scores

Convert vendor evaluation scores into a single risk score (1-5 scale) using the weights.

Logic:
1. For each vendor, compute a weighted average of their scores (0-100 scale)
2. Convert to risk scale: 100 → risk 1.0 (lowest), 0 → risk 5.0 (highest)
   Formula: `risk = 5 - (weighted_score / 100) * 4`


In [ ]:
def calculate_risk_scores(vendor_data, weights):
    """Calculate weighted risk scores for each vendor. Returns list of scores (1-5)."""
    risk_scores = []
    for _, row in vendor_data.iterrows():
        # Weighted average across all criteria (0-100 scale)
        weighted_score = sum(row[metric] * weight for metric, weight in weights.items())
        # Convert to 1-5 risk scale: higher score = lower risk
        risk_level = 5 - (weighted_score / 100) * 4
        risk_scores.append(round(risk_level, 2))
    return risk_scores

vendors_df['Risk_Score'] = calculate_risk_scores(vendors_df, scoring_weights)
print('Vendor Risk Scores:')
print(vendors_df[['Vendor', 'Risk_Score']].to_string(index=False))


## Step 4: Vendor Recommendation Logic

Based on risk scores and SLA compliance, assign a recommendation category for each vendor.

Governance thresholds (aligned with BNM RMiT outsourcing risk tiers):
- Risk ≤ 1.5 AND SLA compliant → **"Highly Recommended"**
- Risk ≤ 2.5 AND SLA compliant → **"Recommended"**
- Risk ≤ 3.5 → **"Acceptable (with monitoring)"**
- Risk > 3.5 → **"Not Recommended"**


In [ ]:
# SLA compliance thresholds (from SLA Requirements sheet)
uptime_threshold  = 99.9    # % - BNM RMiT minimum for critical systems
latency_threshold = 800     # ms - PrecisionRisk AI SLA requirement

compliance_df = sla_df.groupby('Vendor').agg(
    Avg_Uptime=('Uptime', 'mean'),
    Avg_Latency=('Latency_ms', 'mean')
).round(3).reset_index()

compliance_df['SLA_Compliant'] = (
    (compliance_df['Avg_Uptime']  >= uptime_threshold) &
    (compliance_df['Avg_Latency'] <= latency_threshold)
)

# Merge risk scores with compliance
summary = vendors_df[['Vendor', 'Risk_Score']].merge(compliance_df, on='Vendor')

def get_recommendation(risk_score, sla_compliant):
    """Return recommendation string based on risk score and SLA compliance."""
    if risk_score <= 1.5 and sla_compliant:
        return "Highly Recommended"
    elif risk_score <= 2.5 and sla_compliant:
        return "Recommended"
    elif risk_score <= 3.5:
        return "Acceptable (with monitoring)"
    else:
        return "Not Recommended"

summary['Recommendation'] = summary.apply(
    lambda row: get_recommendation(row['Risk_Score'], row['SLA_Compliant']), axis=1
)

print('Vendor Assessment Summary:')
print(summary.to_string(index=False))


## Step 4.5: Scenario Analysis - Model Update Incident

The PrecisionRisk AI incident involved an unannounced vendor model update that caused false claim denials - a breach of BNM RMiT change management requirements and potentially the PDPA 2024 accuracy principle.

**Which vendors carry the highest risk around model update notification and change management?**


In [ ]:
# Model update risk analysis
# Key signals: Model_Transparency (low = less visibility into updates)
#              AI_Model_Governance (low = weaker change management controls)
#              Exit_Strategy (low = harder to switch if update causes issues)

model_update_risk = vendors_df[['Vendor', 'Model_Transparency', 'AI_Model_Governance', 'Exit_Strategy']].copy()
model_update_risk['Update_Risk_Score'] = (
    (100 - model_update_risk['Model_Transparency'])  * 0.40 +
    (100 - model_update_risk['AI_Model_Governance']) * 0.40 +
    (100 - model_update_risk['Exit_Strategy'])        * 0.20
)

model_update_risk = model_update_risk.sort_values('Update_Risk_Score', ascending=False)

print("Model Update Notification Risk Assessment")
print("=" * 65)
for _, row in model_update_risk.iterrows():
    score = row['Update_Risk_Score']
    level = "HIGH" if score >= 40 else ("MEDIUM" if score >= 25 else "LOW")
    print(f"  {row['Vendor']:<20} Update Risk: {level:<8} (score: {score:.1f})")
    if level == "HIGH":
        print(f"    ⚠ Justification: Low transparency ({row['Model_Transparency']}) or weak AI governance")
        print(f"    ⚠ BNM RMiT Action: Require contractual 14-day advance model update notice")
    elif level == "MEDIUM":
        print(f"    → Monitor: Request model change logs quarterly")
    else:
        print(f"    ✓ Well-governed: Strong transparency and AI model governance scores")
    print()


## Step 5: Risk Comparison Chart

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sorted_v = vendors_df.sort_values('Risk_Score')
colors = ['#2ecc71' if s <= 1.5 else '#f39c12' if s <= 3 else '#e74c3c'
          for s in sorted_v['Risk_Score']]

bars = ax.barh(sorted_v['Vendor'], sorted_v['Risk_Score'], color=colors, edgecolor='white', linewidth=0.8)
ax.set_xlabel('Risk Score (1=Lowest, 5=Highest)', fontsize=12, fontweight='bold')
ax.set_title('Vendor Risk Assessment - PrecisionRisk AI\n(BNM RMiT Aligned)', fontsize=14, fontweight='bold')
ax.set_xlim(0, 5)
ax.axvline(x=1.5, color='green',  linestyle='--', alpha=0.5, label='Highly Recommended threshold')
ax.axvline(x=2.5, color='orange', linestyle='--', alpha=0.5, label='Recommended threshold')
ax.axvline(x=3.5, color='red',    linestyle='--', alpha=0.5, label='Acceptable threshold')
ax.legend(fontsize=9)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.05, bar.get_y() + bar.get_height()/2,
            f'{w:.2f}', ha='left', va='center', fontweight='bold', fontsize=10)

ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('vendor_risk_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved: vendor_risk_comparison.png")


## Step 6: SLA Compliance Dashboard

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

vendor_colors = {
    'ClarityCore API': '#3498db',
    'ArcVault AI':     '#2ecc71',
    'NexaScale AI':    '#9b59b6',
    'PrismLogic AI':   '#e74c3c',   # red - poor performer
    'SentinelIQ AI':   '#f39c12',
}

for vendor in vendors_df['Vendor']:
    vd = sla_df[sla_df['Vendor'] == vendor].sort_values('Day')
    ax1.plot(vd['Day'], vd['Uptime'], marker='o', label=vendor,
             linewidth=1.5, markersize=3, color=vendor_colors[vendor])
ax1.axhline(y=uptime_threshold, color='red', linestyle='--', linewidth=2,
            label=f'SLA Threshold ({uptime_threshold}%)')
ax1.set_ylabel('Uptime (%)', fontweight='bold')
ax1.set_title('30-Day Uptime Tracking - PrecisionRisk AI Vendors', fontsize=13, fontweight='bold')
ax1.legend(loc='lower left', fontsize=9)
ax1.set_ylim(98, 100.2)
ax1.grid(True, alpha=0.3)

for vendor in vendors_df['Vendor']:
    vd = sla_df[sla_df['Vendor'] == vendor].sort_values('Day')
    ax2.plot(vd['Day'], vd['Latency_ms'], marker='o', label=vendor,
             linewidth=1.5, markersize=3, color=vendor_colors[vendor])
ax2.axhline(y=latency_threshold, color='red', linestyle='--', linewidth=2,
            label=f'SLA Threshold ({latency_threshold}ms)')
ax2.set_xlabel('Day', fontweight='bold')
ax2.set_ylabel('Latency (ms)', fontweight='bold')
ax2.set_title('30-Day Latency Tracking', fontsize=13, fontweight='bold')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sla_compliance_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("Dashboard saved: sla_compliance_dashboard.png")


## Step 7: Radar Chart - Multi-Dimensional Comparison

In [ ]:
top3 = vendors_df.nsmallest(3, 'Risk_Score')
categories = ['Security', 'Data Handling', 'Transparency', 'Financial', 'SLA', 'Exit Strategy', 'AI Governance']
N = len(categories)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), subplot_kw=dict(projection='polar'))
colors = ['#3498db', '#2ecc71', '#e74c3c']

for idx, (_, row) in enumerate(top3.iterrows()):
    ax = axes[idx]
    values = [
        row['Security_Score'], row['Data_Handling_Score'], row['Model_Transparency'],
        row['Financial_Stability'], row['SLA_Compliance'], row['Exit_Strategy'], row['AI_Model_Governance']
    ]
    values += values[:1]  # close the polygon
    angles = [n / N * 2 * pi for n in range(N)]
    angles += angles[:1]

    ax.plot(angles, values, 'o-', linewidth=2, color=colors[idx])
    ax.fill(angles, values, alpha=0.25, color=colors[idx])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=8)
    ax.set_ylim(0, 100)
    ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_yticklabels(['20','40','60','80','100'], fontsize=7)
    ax.set_title(f"{row['Vendor']}\nRisk: {row['Risk_Score']}", fontsize=11, fontweight='bold', pad=20)
    ax.grid(True, alpha=0.3)

plt.suptitle('Multi-Dimensional Vendor Comparison (Top 3 by Risk Score)\nPrecisionRisk AI - BNM RMiT Assessment',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('vendor_radar.png', dpi=150, bbox_inches='tight')
plt.show()
print("Radar chart saved: vendor_radar.png")


## Step 8: Final Assessment Report

In [ ]:
print('='*90)
print('VENDOR ASSESSMENT REPORT - PrecisionRisk AI Claims Processing System')
print('Regulatory Framework: BNM RMiT | AIGE Guidelines | PDPA 2024 | FSA/IFSA 2013')
print('='*90)

for _, row in summary.sort_values('Risk_Score').iterrows():
    print(f"\n  {row['Vendor']}")
    print(f"     Risk Score:      {row['Risk_Score']:.2f}/5.0")
    print(f"     Avg Uptime:      {row['Avg_Uptime']:.3f}% {'✓' if row['Avg_Uptime'] >= uptime_threshold else '✗ BREACH'}")
    print(f"     Avg Latency:     {row['Avg_Latency']:.1f}ms {'✓' if row['Avg_Latency'] <= latency_threshold else '✗ BREACH'}")
    print(f"     SLA Compliant:   {'Yes ✓' if row['SLA_Compliant'] else 'No ✗'}")
    print(f"     Recommendation:  {row['Recommendation']}")

print('\n' + '='*90)
print(f"Scoring weights used: {scoring_weights}")
print(f"SLA thresholds: Uptime >= {uptime_threshold}%, Latency <= {latency_threshold}ms")
print('\nRegulatory Notes:')
print('  • PDPA 2024: All vendors with Full data access are direct processors - subject to RM1M liability')
print('  • BNM RMiT: Vendors rated Acceptable or below require quarterly review and documented risk acceptance')
print('  • AIGE: PrismLogic AI Low transparency score is incompatible with takaful Shariah audit requirements')


## Step 9: Exit Strategy Analysis - Vendor Lock-in

For GenAI vendors, "exit strategy" extends beyond simple data portability. Under **BNM RMiT**, material outsourcing arrangements must have documented and *tested* exit plans.


In [ ]:
# Bucket vendors by exit-strategy switching cost
# Exit_Strategy score: higher = easier/cheaper to exit

def switching_cost_bucket(score):
    if score >= 88:
        return "Low switching cost"
    elif score >= 75:
        return "Moderate switching cost"
    else:
        return "High switching cost"

vendors_df['Switching_Cost'] = vendors_df['Exit_Strategy'].apply(switching_cost_bucket)

print("Exit Strategy - Vendor Lock-in Analysis")
print("(BNM RMiT requires documented and tested exit plans for all material vendors)")
print("=" * 65)

for bucket in ["Low switching cost", "Moderate switching cost", "High switching cost"]:
    vendors_in_bucket = vendors_df[vendors_df['Switching_Cost'] == bucket]['Vendor'].tolist()
    icon = "✓" if "Low" in bucket else ("→" if "Moderate" in bucket else "⚠")
    print(f"\n{icon} {bucket}:")
    for v in vendors_in_bucket:
        score = vendors_df[vendors_df['Vendor']==v]['Exit_Strategy'].values[0]
        print(f"    - {v} (Exit Score: {score})")
    if "High" in bucket:
        print("    BNM RMiT Action Required: Document exit plan, test annually, escalate to board risk committee")
    elif "Moderate" in bucket:
        print("    BNM RMiT Action: Maintain documented exit plan, review semi-annually")


## Step 10: Sensitivity Analysis (Stretch Goal)

Small changes in vendor scores can flip recommendation tiers. This matters for BNM RMiT - a vendor that sits near a threshold needs documented rationale for its tier placement.


In [ ]:
# Sensitivity analysis on PrismLogic AI - the borderline vendor
target_vendor = 'PrismLogic AI'

print(f"Sensitivity Analysis: {target_vendor}")
print("=" * 65)

original_row = vendors_df[vendors_df['Vendor'] == target_vendor].copy()
original_score = calculate_risk_scores(original_row, scoring_weights)[0]
original_rec   = get_recommendation(original_score,
    compliance_df[compliance_df['Vendor']==target_vendor]['SLA_Compliant'].values[0])

print(f"\nOriginal scores:")
for col in scoring_weights.keys():
    print(f"  {col:<25} {original_row[col].values[0]}")
print(f"\n  → Risk Score: {original_score}  |  Recommendation: {original_rec}")

for delta in [+10, -10]:
    modified = vendors_df.copy()
    for col in scoring_weights.keys():
        modified.loc[modified['Vendor']==target_vendor, col] = (
            modified.loc[modified['Vendor']==target_vendor, col] + delta
        ).clip(0, 100)

    new_score = calculate_risk_scores(
        modified[modified['Vendor']==target_vendor], scoring_weights)[0]
    sla_ok = compliance_df[compliance_df['Vendor']==target_vendor]['SLA_Compliant'].values[0]
    new_rec = get_recommendation(new_score, sla_ok)
    direction = "improved" if delta > 0 else "degraded"

    print(f"\nIf all scores {direction} by {abs(delta)} points:")
    print(f"  → Risk Score: {new_score}  |  Recommendation: {new_rec}")
    if new_rec != original_rec:
        print(f"  ⚠ TIER CHANGE: '{original_rec}' → '{new_rec}'")
        print(f"  BNM RMiT implication: Risk acceptance decision must be re-escalated to CRO")

print("\nConclusion: Small score changes can flip the recommendation tier.")
print("This reinforces BNM RMiT's requirement for documented, evidence-based vendor scoring.")
